# L39 · 亲手写 FFT — Cooley-Tukey 实现

**目标**：手写 Cooley-Tukey 递归 FFT，让 `tests/audio/test_transforms.py` 全绿。

🔗 **Aurora 连接**：本节实现镜像 `src/aurora/audio/transforms.py` 的 `fft()` 函数。

FFT 的本质是**分治**：把 N 点 DFT 拆成两个 N/2 点 DFT，再用蝶形公式合并，把 O(N²) 变成 O(N log N)。每层递归只做旋转因子乘法和加减，整个计算树有 log₂N 层，每层 N 次运算。从零实现一遍，你对 STFT、mel 滤波器的数值行为就有了直觉基础。

In [ ]:
import numpy as np
import time
from aurora.audio.transforms import fft as aurora_fft

## 1. 递归结构：蝶形合并

Cooley-Tukey 核心恒等式：

```
X[k]     = E[k] + exp(-2πi·k/N) · O[k]      (k = 0..N/2-1)
X[k+N/2] = E[k] - exp(-2πi·k/N) · O[k]      (k = 0..N/2-1)
```

其中 `E = fft(x[::2])`（偶数下标），`O = fft(x[1::2])`（奇数下标）。
旋转因子 `exp(-2πi·k/N)` 乘完就是「蝶形（butterfly）」操作；`X[k+N/2]` 只是符号翻转，**不需要额外乘法**。
递归基：`N==1` 时 DFT 就是原值本身，直接返回。

In [ ]:
# 演示：手算 N=2 的 DFT vs FFT 公式
x2 = np.array([3.0, 7.0])
# DFT 定义: X[k] = sum_n x[n]*exp(-2pi*i*k*n/N)
X2_def = np.array([
    x2[0] + x2[1],           # k=0: exp(0)=1
    x2[0] - x2[1],           # k=1: exp(-pi*i)=-1
], dtype=complex)
X2_np = np.fft.fft(x2)
print('手算 X[0]:', X2_def[0], '  numpy:', X2_np[0])
print('手算 X[1]:', X2_def[1], '  numpy:', X2_np[1])
print('误差:', np.max(np.abs(X2_def - X2_np)))

## 2. 必须是 2 的幂；非 2 的幂用补零

Cooley-Tukey 每步把 N 对半分，要求 N 是 2 的幂次（`N = 2^k`）。
对任意长度 `L` 的信号，标准做法是找最小的 `N = 2^ceil(log2(L))`，在末尾补零到 N，变换后截取前 L 个系数——频率分辨率不变，只是增加了频率网格密度（细化）。

```
N_padded = 2 ** int(np.ceil(np.log2(len(x))))
x_padded = np.zeros(N_padded, dtype=complex)
x_padded[:len(x)] = x
```

In [ ]:
# 演示：N=5 补零到 8
x5 = np.array([1, 2, 3, 4, 5], dtype=float)
N_padded = 2 ** int(np.ceil(np.log2(len(x5))))
x_pad = np.zeros(N_padded)
x_pad[:len(x5)] = x5
print(f'原始长度: {len(x5)}  补零后: {N_padded}')
print('补零后:', x_pad)
print('np.fft.fft(补零后):', np.round(np.fft.fft(x_pad), 4))

## 3. 数值误差：浮点累积 < 1e-10

递归 FFT 有 `log₂N` 层，每层做浮点乘法；误差随层数线性累积，量级约 `N·log₂N·ε_machine`。
对 `N=1024`，double 精度下误差约 `1024·10·2.2e-16 ≈ 2e-12`，远低于 `1e-10`。
验证工具：`np.testing.assert_allclose(my_result, reference, atol=1e-9)`，失败时会打印最大偏差位置。

```
# atol 是绝对容忍，rtol 是相对容忍；对复数频谱两者都要设
np.testing.assert_allclose(A, B, rtol=1e-9, atol=1e-9)
```

In [ ]:
# 演示：量化不同 N 下的浮点误差
print(f'{'N':>6}  {'max_abs_error':>18}')
for N in [4, 16, 64, 256, 1024, 4096]:
    x = np.random.randn(N)
    ref = np.fft.fft(x)
    # 用 numpy 自身做对照（后面换成 my_fft）
    err = np.max(np.abs(ref - np.fft.fft(x)))  # 占位，替换后误差才有意义
    print(f'{N:>6}  {err:>18.2e}')
print('\n(以上对照 numpy 自身，误差为 0；实现 my_fft 后替换左侧再运行)')

## 4. ✏️ 实现 `my_fft(x)`

**推理路线**：
1. 若 `len(x) == 1`，直接返回 `x.astype(complex)`（递归基）
2. 递归：`E = my_fft(x[::2])`，`O = my_fft(x[1::2])`
3. 构造旋转因子：`twiddle = exp(-2πi · np.arange(N//2) / N)`
4. 蝶形：`top = E + twiddle * O`；`bottom = E - twiddle * O`
5. 返回 `np.concatenate([top, bottom])`

**参考输入输出**：
```
x = np.array([1, 2, 3, 4, 3, 2, 1, 0], dtype=float)
my_fft(x)   # 应与下列结果误差 < 1e-10
np.fft.fft(x)
# [16.+0.j, -4.+0.j, 0.-0.j, -0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, -4.-0.j]
# (数值经四舍五入展示)
```

<details><summary>点击查看参考实现</summary>

```python
def my_fft(x: np.ndarray) -> np.ndarray:
    N = len(x)
    if N == 1:
        return x.astype(complex)
    E = my_fft(x[::2])            # 偶数下标子问题
    O = my_fft(x[1::2])           # 奇数下标子问题
    k = np.arange(N // 2)
    twiddle = np.exp(-2j * np.pi * k / N)  # 旋转因子
    top    = E + twiddle * O
    bottom = E - twiddle * O
    return np.concatenate([top, bottom])
```

</details>

In [ ]:
def my_fft(x: np.ndarray) -> np.ndarray:
    """递归 Cooley-Tukey radix-2 FFT。x 长度必须是 2 的幂。"""
    N = len(x)
    # ✏️ TODO 1: 递归基 — N==1 时直接返回

    # ✏️ TODO 2: 递归拆分偶/奇下标
    E = ...  # my_fft(x[::2])
    O = ...  # my_fft(x[1::2])

    # ✏️ TODO 3: 旋转因子 twiddle = exp(-2pi*i * k / N), k = 0..N/2-1
    k = np.arange(N // 2)
    twiddle = ...

    # ✏️ TODO 4: 蝶形合并
    top    = ...  # E + twiddle * O
    bottom = ...  # E - twiddle * O

    # ✏️ TODO 5: 拼接返回
    return ...

In [ ]:
# ── 正确性检查 ──────────────────────────────────────────────
x_demo = np.array([1, 2, 3, 4, 3, 2, 1, 0], dtype=float)
ref_demo = np.fft.fft(x_demo)
out_demo = my_fft(x_demo)
print('my_fft([1,2,3,4,3,2,1,0]):'); print(np.round(out_demo, 6))
print('np.fft.fft:');               print(np.round(ref_demo, 6))
print('最大误差:', np.max(np.abs(out_demo - ref_demo)))

all_pass = True
for N in [4, 8, 16, 64, 256]:
    x = np.random.randn(N)
    if not np.allclose(my_fft(x), np.fft.fft(x), atol=1e-9):
        print(f'✗ N={N} 失败')
        all_pass = False
if all_pass:
    print('\n✅ 所有 N in [4,8,16,64,256] 通过，误差 < 1e-9')

## 5. 参数实验：速度对比

- `N = 512`：naive DFT（O(N²)) vs `my_fft`（O(N log N)）vs `np.fft.fft`（C 扩展）
- 预期：`my_fft` 比 naive DFT 快约 `log₂(512) = 9` 倍；`np.fft.fft` 比 `my_fft` 快 10-50 倍（C 扩展 + SIMD）
- 调整 `N` 观察：倍数比例随 `N` 增大而增大（log 增益更显著）

In [ ]:
def naive_dft(x: np.ndarray) -> np.ndarray:
    """暴力 O(N²) DFT，用于速度基准对比。"""
    N = len(x)
    n = np.arange(N)
    k = n.reshape((N, 1))
    M = np.exp(-2j * np.pi * k * n / N)
    return M @ x.astype(complex)

N = 512
x_bench = np.random.randn(N)

reps = 20
t0 = time.perf_counter()
for _ in range(reps): naive_dft(x_bench)
t_naive = (time.perf_counter() - t0) / reps * 1000

t0 = time.perf_counter()
for _ in range(reps): my_fft(x_bench)
t_my = (time.perf_counter() - t0) / reps * 1000

t0 = time.perf_counter()
for _ in range(reps): np.fft.fft(x_bench)
t_np = (time.perf_counter() - t0) / reps * 1000

print(f'N={N}  (平均 {reps} 次，单位 ms)')
print(f'  naive_dft : {t_naive:8.3f} ms')
print(f'  my_fft    : {t_my:8.3f} ms   ({t_naive/t_my:.1f}× faster than naive)')
print(f'  np.fft.fft: {t_np:8.3f} ms   ({t_my/t_np:.1f}× faster than my_fft)')
print(f'\n理论加速比 naive→FFT: log₂({N}) = {np.log2(N):.1f}')

## 收束

`my_fft(x)` 通过递归蝶形操作输出 N 个复数频谱系数，误差控制在 `1e-9` 以内，与 `aurora.audio.transforms.fft` 在数值上等价。该函数直接服务于 `src/aurora/audio/transforms.py`，是 STFT（短时傅里叶变换）的内层调用——下一节 Day 4 将在滑动窗口上批量调用它，输出时频谱图（spectrogram）。